Script to **create synthetic data** (with settings defined in the settings file)


All rights reserved. Material from GEOFCAN 2025 Workshop led by Coline Mollaret

In [ ]:
import numpy as np
import pygimli as pg
import pygimli.meshtools as mt
from pygimli.physics import ERTManager
from pygimli.physics import TravelTimeManager
from pygimli.physics.traveltime import createRAData
from pygimli.physics import ert

#############################################
# to find "invlib" in the main folder
#import sys, os
#sys.path.insert(0, os.path.abspath("../.."))
#path_SYN_data = os.path.abspath("../../..") #to find the folder where the data are stored
#############################################


#############################################
import sys, os
# try and except functions to change the directory only once to stay in the right folder even the script/cell is run again
try:  
    findpaths
except NameError:  
    os.chdir(os.path.abspath("..")) #to find the folder where the data are stored
    findpaths=True
print(os.getcwd()) #print the current working directory
#############################################

### CASE ######################################################################
case='3L'
#case='3L+block'

# Import profile specific settings if they exist, otherwise import default settings:
if os.path.exists('profile_settings/'+case+'_settings.py'):
    exec(open('profile_settings/'+case+'_settings.py', 'r').read()) # reads case specific settings
    print('case settings exist and were loaded')
else:
    print('case settings do not exist, default settings are used')
    
    ### DEFAULT SETTINGS ######################################################
    ERT_config = 'dd' #known schemes ['wa', 'wb', 'pp', 'pd', 'dd', 'slm', 'hw', 'gr']
    if ERT_config == 'dd': 
        spacing=[1,3,5,7,9] #only for dd

    quality = 33.8
    paraDX = 0.1
    paraMaxCellSize = 6
    paraBoundary = 1 #Offset to the parameter domain boundary in absolute sensor spacing
    boundary = 1 #Boundary width to be appended for domain prolongation in absolute para domain width. Values lower than 0 force the boundary to be 4 times para domain width
    boundaryMaxCellSize = 70
    noiselevelERT = 0.025 # add relative noise to the simulated data. noiseLevel*100 in % (0.025 = 2.5%)
    noiseAbsERT = 0.0
    noiselevelRST = 0.025
    errorRST = 0.0004    # add absolute noise to the simulated data in ms
    errorERT = 0.01    # add absolute noise to the simulated data in % (default ERT: 0.01)
    showmesh=0
    ### END OF DEFAULT SETTINGS ###############################################


mesh = pg.load('results/mesh/'+case+'_mesh.bms')
sensors = np.load('results/mesh/'+case+'_sensors.npy', allow_pickle=True)
rhotrue = np.loadtxt('data/'+case+'_rhotrue.dat')
veltrue = np.loadtxt('data/'+case+'_veltrue.dat')
shots = np.load('results/mesh/'+case+'_shots.npy', allow_pickle=True)


pg.boxprint("Simulate apparent resistivities")

# Create more realistic data set
# ertScheme = ert.createData(sensors, "dd", spacings=[1,2,4])
if ERT_config == 'dd': 
    ertScheme = ert.createData(sensors, ERT_config, spacings=spacing)
else: 
    ertScheme = ert.createData(sensors, ERT_config)
k = ert.geometricFactors(ertScheme)
ertScheme.markInvalid(pg.abs(k) > 5000)
ertScheme.removeInvalid()

ert = ERTManager()

# Create suitable mesh for ert forward calculation
# NOTE: In the published results paraMaxCellSize=1.0 was used, which is
# increased here to allow testing on Continuous Integration services.
meshERTFWD = mt.createParaMesh(ertScheme, quality=quality, paraDX=paraDX,
                               paraMaxCellSize=paraMaxCellSize,
                               boundaryMaxCellSize=boundaryMaxCellSize,
                               paraBoundary=paraBoundary,boundary=boundary)
if showmesh == 1:
    pg.show(meshERTFWD)

res = pg.Vector()
pg.interpolate(mesh, rhotrue, meshERTFWD.cellCenters(), res)
res = mt.fillEmptyToCellArray(meshERTFWD, res, slope=True)
ert.setMesh(meshERTFWD)
ert.fop.createRefinedForwardMesh()
ertData = ert.simulate(mesh=meshERTFWD, scheme=ertScheme, res=res, noiseLevel=noiselevelERT,
                       noiseAbs=noiseAbsERT)
# ertData.set("err", np.ones(ertData.size()) * errorERT) #test to do the same as for RST
ertData.save('data/'+case+'_erttrue.dat')
ertData.save('data/'+case+'_rhoa.txt')

ert.setData(ertData)
ert.setMesh(meshERTFWD)
ert.inv.dataVals = ertData("rhoa")

pg.boxprint("Simulate traveltimes")
meshRSTFWD = pg.Mesh()
meshRSTFWD.createMeshByMarker(meshERTFWD, 2)

vel = pg.Vector()
pg.interpolate(mesh, veltrue, meshRSTFWD.cellCenters(), vel)
vel = mt.fillEmptyToCellArray(meshRSTFWD, vel, slope=False) #default

# ttScheme = createRAData(sensors)
ttScheme = createRAData(shots)
rst = TravelTimeManager(verbose=True)

# meshRSTFWD.createSecondaryNodes(3) #secondoes were already implemented before
ttData = rst.simulate(mesh=meshRSTFWD, slowness=1./vel, scheme=ttScheme,
                      noisify=True, noiseLevel=noiselevelRST, noiseAbs=0.0)
ttData.set("err", np.ones(ttData.size()) * errorRST)

rst.setData(ttData)
rst.fop.data.save('data/'+case+'_tttrue.dat')
rst.fop.data.save('data/'+case+'_tt.txt')
